# Extract frozen GLIM EEG tokens + pooled vectors, and the Gate-1 verdict

Route B, Gate 1 (D-027). **GPU.** Attach the canonical sharded dataset
`thestonedape/task-aware-eegtotext` (v1) and the checkpoint
`thestonedape/glim-zuco-checkpoint`; enable Internet and the secret `GITHUB_TOKEN`.

**One complete run** (no pre-flight, no sample). It extracts, for the full
primary ZuCo2 NR/TSR cohort (9,011 trials), BOTH the 96 unpooled `eeg_tokens`
[96,1024] and GLIM's learned pooled `eeg_vector` [1024] from the identical
prompt-neutral (`all_masked`) forward pass that produced P4b's pooled vectors,
then runs the token-collapse diagnostic over the whole cohort and prints a
**COLLAPSED / WEAK / RICH** verdict. Held-out test never touched.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
COMMIT = '552258e3c310039765e9c61fd36d07c7f6c09824'
WORKTREE = '/kaggle/working/SemKey'
GLIM_REPO_URL = 'https://github.com/justin-xzliu/GLIM.git'
GLIM_COMMIT = 'e1f202cb793cfe7292fbc0072a4c26a7dd0660d9'
GLIM_WORKTREE = '/kaggle/working/GLIM'
EXPECTED_INDEX_SHA256 = 'bdaaaf5c91d3c9eec16a0727825da996fd2186867245951bfdfdc92aab7738b0'
CHECKPOINT_SHA256 = '25fcd31d1d6cafc9a0656c50a4916ba6ee106884b269d347284784cc0522c8ba'
OUTPUT = '/kaggle/working/task-aware-eeg2text-glim-tokens'
BATCH_SIZE = 16
CHUNK_SIZE = 128
DTYPE = 'float16'
assert len(COMMIT) == len(GLIM_COMMIT) == 40
assert all(len(v) == 64 for v in (EXPECTED_INDEX_SHA256, CHECKPOINT_SHA256))

In [ ]:
import glob, hashlib, json, os, platform, shutil, subprocess, sys, torch
from kaggle_secrets import UserSecretsClient
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
print({'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0)})
github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as handle:
    handle.write("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n")
os.chmod(askpass, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
for path in (WORKTREE, GLIM_WORKTREE):
    if os.path.exists(path):
        shutil.rmtree(path)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=clone_env)
finally:
    os.remove(askpass)
    del github_token, clone_env
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', COMMIT], check=True)
assert subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip() == COMMIT
subprocess.run(['git', 'clone', GLIM_REPO_URL, GLIM_WORKTREE], check=True)
subprocess.run(['git', '-C', GLIM_WORKTREE, 'checkout', '--detach', GLIM_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', GLIM_WORKTREE, 'rev-parse', 'HEAD'], text=True).strip() == GLIM_COMMIT
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'lightning==2.4.0', 'torchmetrics==1.3.1', 'einops==0.8.0', 'timm==0.9.16', 'transformers==4.52.0'], check=True)
env = os.environ.copy(); env['PYTHONPATH'] = WORKTREE
for t in ('evaluation.test_token_late_interaction', 'evaluation.test_token_diagnostics', 'evaluation.test_extract_frozen_glim_tokens'):
    subprocess.run([sys.executable, '-B', '-m', 'unittest', t], check=True, cwd=WORKTREE, env=env)
print({'clone': 'PASS', 'token_tests': 'PASS'})

In [ ]:
manifest_paths = glob.glob('/kaggle/input/**/metadata/shard_manifest.json', recursive=True)
assert len(manifest_paths) == 1, ('Attach exactly one canonical sharded dataset', manifest_paths)
dataset_root = os.path.dirname(os.path.dirname(manifest_paths[0]))
checkpoint_paths = glob.glob('/kaggle/input/**/*.ckpt', recursive=True)
assert len(checkpoint_paths) == 1, ('Attach exactly one GLIM checkpoint', checkpoint_paths)
checkpoint = checkpoint_paths[0]
state = hashlib.sha256()
with open(checkpoint, 'rb') as handle:
    for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
        state.update(block)
assert state.hexdigest() == CHECKPOINT_SHA256, 'checkpoint SHA mismatch'
print({'dataset_root': dataset_root, 'checkpoint': checkpoint})

In [ ]:
# ONE complete run: full 9,011-trial cohort. Resumable -- hash-valid chunks are
# reused, so re-run this cell after any interruption. Prints the Gate-1 verdict.
cmd = [
    sys.executable, '-B', os.path.join(WORKTREE, 'evaluation', 'extract_frozen_glim_tokens.py'),
    '--dataset-root', dataset_root, '--output-root', OUTPUT, '--glim-root', GLIM_WORKTREE,
    '--checkpoint', checkpoint, '--glim-commit', GLIM_COMMIT,
    '--expected-index-sha256', EXPECTED_INDEX_SHA256,
    '--expected-checkpoint-sha256', CHECKPOINT_SHA256,
    '--device', 'cuda', '--batch-size', str(BATCH_SIZE), '--chunk-size', str(CHUNK_SIZE),
    '--dtype', DTYPE,
]
subprocess.run(cmd, check=True, cwd=WORKTREE, env={**os.environ, 'PYTHONPATH': WORKTREE})

In [ ]:
import numpy as np
sys.path.insert(0, WORKTREE)
from evaluation.extract_frozen_glim_tokens import token_chunk_sha256
index = json.load(open(os.path.join(OUTPUT, 'token_index.json'), encoding='utf-8'))
assert index['token_shape'] == [96, 1024] and index['dtype'] == DTYPE
assert index['prompt_mode'] == 'all_masked' and index['glim_commit'] == GLIM_COMMIT
assert index['total_rows'] == 9011, index['total_rows']
# Integrity: reload each chunk's arrays + identity and recompute the array-bytes
# hash (NOT the .npz file bytes); confirm it matches the index and per-chunk meta.
for entry in index['chunks']:
    npz = os.path.join(OUTPUT, entry['token_file'])
    meta = json.load(open(npz[:-4] + '.json', encoding='utf-8'))
    with np.load(npz) as arch:
        assert set(arch.files) == {'tokens', 'vectors'}, arch.files
        recomputed = token_chunk_sha256(
            arch['tokens'], arch['vectors'],
            meta['trial_ids'], meta['sample_ids'], meta['source_dataframe_row_indices'])
    assert recomputed == entry['sha256'] == meta['sha256'], entry['token_file']
diag = json.load(open(os.path.join(OUTPUT, 'gate1_token_diagnostic.json'), encoding='utf-8'))
run_metadata = {
    'status': 'pass', 'project_commit': COMMIT, 'glim_commit': GLIM_COMMIT,
    'checkpoint_sha256': CHECKPOINT_SHA256, 'dataset_index_sha256': EXPECTED_INDEX_SHA256,
    'total_rows': index['total_rows'], 'token_shape': index['token_shape'], 'dtype': DTYPE,
    'combined_chunk_sha256': index['combined_chunk_sha256'], 'gate1_verdict': diag['verdict'],
    'python': platform.python_version(), 'torch': torch.__version__,
    'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0),
}
with open(os.path.join(OUTPUT, 'run_metadata.json'), 'w', encoding='utf-8') as handle:
    json.dump(run_metadata, handle, indent=2, sort_keys=True); handle.write('\n')
for path in (WORKTREE, GLIM_WORKTREE):
    if os.path.exists(path):
        shutil.rmtree(path)
print({'rows': index['total_rows'], 'chunks': index['num_chunks'],
       'combined_chunk_sha256': index['combined_chunk_sha256']})
print('redundancy raw / centered:', round(diag['within_trial_redundancy_mean'], 4),
      '/', round(diag['within_trial_redundancy_centered_mean'], 4),
      '| eff-rank frac:', round(diag['effective_rank_fraction_of_T'], 4))
print('GATE-1 VERDICT (advisory):', diag['verdict'])
print('GLIM TOKEN EXTRACTION: PASS')

After PASS, save the printed OUTPUT (`task-aware-eeg2text-glim-tokens`) as a new
**private** Kaggle dataset, Version 1 (contains tokens [96,1024], pooled vectors
[1024], the index, run_metadata, and gate1_token_diagnostic.json).

**The Gate-1 verdict is advisory** (feasibility only, not model selection). RICH
-> proceed to Gate 2 (freeze the pooled-vs-token design). COLLAPSED / WEAK ->
inspect before falling back to Route A: if the raw within-trial redundancy is high
but the *centered* value is much lower, the tokens are anisotropic (a shared DC
direction), not collapsed. Report the numbers; do not build the training loop
before the verdict is recorded.